# MVP – Übergabe-Check Brücken (Entwurf)
Vom BIM-Projekt ins Erhaltungsmanagement · Team 1, bxmid03 HS26

Das Notebook setzt den Soll-Prozess aus `workflow/README.md` um. «Laufzeit → Alle ausführen» genügt.

| Workflow-Schritt | Zelle |
|---|---|
| 3 IDS bereitstellen | 4 (Regeln aus GitHub) |
| 5 IFC einreichen | 3 (manuell: Upload / Drive) |
| 6 IDS-Prüfung (ifctester) | 5 |
| 7 Entscheid «Daten vollständig?» + BCF | 6, 7 |
| 8 Stichprobe Fachplaner | 8 (Liste, Beurteilung bleibt manuell) |
| 10 Export Stammdaten + Prüfbericht | 7, 9 |
| Dashboard «Vollständigkeit in % je Bauteilgruppe» | 10 |

Alles ausser dem Modell kommt aus dem GitHub-Repository: `checks/pruefregeln.yaml`, `checks/export_mapping.yaml`.
**Entwurf:** Dateinamen, Regeln und Mapping sind Platzhalter, bis die überarbeitete IDS vorliegt.

## 1 · Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ifctester", "pyyaml", "pandas", "openpyxl", "matplotlib"], check=True)

## 2 · Konfiguration

In [ ]:
#@title Konfiguration { display-mode: "form" }
REPO_URL        = "https://github.com/Raef77/bxmid03-hs26-team-1.git" #@param {type:"string"}
REPO_BRANCH     = "main" #@param {type:"string"}
REGELN_DATEI    = "checks/pruefregeln.yaml" #@param {type:"string"}
IDS_DATEI       = "" #@param {type:"string"}
MAPPING_DATEI   = "checks/export_mapping.yaml" #@param {type:"string"}
MODELL_QUELLE   = "Upload (Desktop)" #@param ["Upload (Desktop)", "Google Drive", "Lokaler Pfad", "Lehrmodell aus Repo"]
MODELL_PFAD     = "" #@param {type:"string"}
EXPORT_BEI_MAENGELN = False #@param {type:"boolean"}
AUSGABE         = "out" #@param {type:"string"}

# IDS_DATEI: optional fertige IDS (.ids/.xml, Pfad relativ zum Repo). Hat Vorrang vor den YAML-Regeln.
# MODELL_PFAD: nur für «Google Drive» und «Lokaler Pfad», z. B. /content/drive/MyDrive/Modell_Wahr.ifc
# Upload: Datei im Dialog wählen oder vorher in den Ordner «modell» der Colab-Dateiansicht ziehen.
from datetime import datetime
from pathlib import Path
JAHR = datetime.now().year
OUT = Path(AUSGABE); OUT.mkdir(exist_ok=True)
print("Prüfjahr:", JAHR)

## 3 · Repository holen und Modell einfügen

In [ ]:
import subprocess, shutil
REPO = Path("repo")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
else:
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO)])
    if r.returncode != 0:
        print("Hinweis: Repo nicht erreichbar (privat oder offline). Es gelten die eingebetteten Standardregeln.")

def modell_beschaffen():
    q = MODELL_QUELLE
    if q == "Lehrmodell aus Repo":
        p = REPO / "models" / "Team_1_Bruecke_BFH_Minor.ifc"
    elif q == "Google Drive":
        from google.colab import drive
        drive.mount("/content/drive")
        p = Path(MODELL_PFAD)
    elif q == "Lokaler Pfad":
        p = Path(MODELL_PFAD)
    else:
        ordner = Path("modell"); ordner.mkdir(exist_ok=True)
        if not list(ordner.glob("*.ifc")):
            from google.colab import files
            for name, daten in files.upload().items():
                (ordner / name).write_bytes(daten)
        gefunden = sorted(ordner.glob("*.ifc"))
        if not gefunden:
            raise FileNotFoundError("Kein IFC hochgeladen. Zelle erneut ausführen und eine .ifc-Datei wählen.")
        p = gefunden[-1]
    if not p.is_file():
        raise FileNotFoundError(f"Modell nicht gefunden: {p}")
    return p

import ifcopenshell
MODELL = modell_beschaffen()
modell = ifcopenshell.open(str(MODELL))
print("Modell:", MODELL.name, "| Schema:", modell.schema, "| Grösse:", round(MODELL.stat().st_size / 1e6, 1), "MB")
for k in ("IfcBridge", "IfcBearing"):
    print(f"  {k}: {len(modell.by_type(k))}")

## 4 · IDS aus GitHub laden (oder aus YAML-Regeln erzeugen)

In [ ]:
import yaml
from ifctester import ids

DEFAULT_REGELN = r"""
# checks/pruefregeln.yaml  (ENTWURF, Struktur vom Notebook gelesen)
# Quelle der Wahrheit für IDS-Regeln, Zusatzprüfungen, Stichprobe und Dashboard-Gruppen.
# Wird die IDS überarbeitet, nur diese Datei anpassen (oder eine fertige .ids-Datei im Notebook angeben).
titel: IDS_Team1 (Entwurf)
ifcVersion: IFC4X3_ADD2

# Platzhalter in Werten: aktuelles_jahr  (wird beim Lauf durch das Jahr ersetzt)
spezifikationen:
  - id: R-01
    name: Lager haben einen Lagertyp
    klasse: IFCBEARING
    anforderungen:
      - pset: BFH_Minor          # offen: CH_Lager oder BFH_Minor / Hersteller-Properties leitend?
        name: Lagertyp
        datentyp: IFCLABEL
        # erlaubte_werte: [Elastomeric]   # laut requirements/ gefordert, laut checks/README nur "vorhanden"
  - id: R-02
    name: Lager haben ein Einbaujahr (ganze Zahl, nicht in der Zukunft)
    klasse: IFCBEARING
    anforderungen:
      - pset: BFH_Minor
        name: Einbaujahr
        datentyp: IFCINTEGER
        min: 1900
        max: aktuelles_jahr
  - id: R-03
    name: Brücke hat ein Baujahr
    klasse: IFCBRIDGE
    anforderungen:
      - pset: CH_Bauwerksdaten
        name: Baujahr
        datentyp: IFCINTEGER
        min: 1800
        max: aktuelles_jahr

# Prüfungen, die IDS nicht abbildet (erscheinen im Bericht und in der Mängelliste, nicht im BCF)
zusatzpruefungen:
  - id: Z-01
    name: Name des Bauteils ist vorhanden und eindeutig
    typ: name_eindeutig
    klassen: [IfcBearing]
    schweregrad: Fehler
  - id: Z-02
    name: Einbaujahr liegt nicht vor dem Baujahr der Brücke
    typ: einbaujahr_nach_baujahr
    klassen: [IfcBearing]
    einbaujahr: BFH_Minor.Einbaujahr
    baujahr: CH_Bauwerksdaten.Baujahr
    schweregrad: Warnung

# Stichprobe für den Fachplaner (Schritt 8, manuell)
stichprobe:
  klassen: [IfcBearing]
  anzahl: 3
  seed: 1
  felder: [Name, BFH_Minor.Lagertyp, BFH_Minor.Lieferant, BFH_Minor.Modell, BFH_Minor.Einbaujahr]

# Dashboard "Vollständigkeit in % je Bauteilgruppe"
gruppen:
  - name: Bauwerk
    klassen: [IfcBridge]
    erforderlich: [CH_Bauwerksdaten.Baujahr]
  - name: Lager
    klassen: [IfcBearing]
    erforderlich: [BFH_Minor.Lagertyp, BFH_Minor.Einbaujahr]
  # - name: Übrige Bauteile      # optional erweitern
  #   klassen: [IfcBeam, IfcColumn, IfcSlab, IfcWall, IfcFooting]
  #   erforderlich: [BFH_Minor.Einbaujahr]

"""
DEFAULT_MAPPING = r"""
# checks/export_mapping.yaml  (ENTWURF)
# Stammdaten-Export für das Erhaltungssystem (z. B. KUBA). Eine Tabelle pro Blatt.
# quelle: GlobalId | Name | Klasse | <Pset>.<Property>
tabellen:
  - blatt: Bauwerk
    klassen: [IfcBridge]
    spalten:
      - {spalte: GlobalId, quelle: GlobalId}
      - {spalte: Bauwerksname, quelle: ASB-ING_2013.Bauwerksname}
      - {spalte: Bauwerksnummer, quelle: ASB-ING_2013.Bauwerksnummer}
      - {spalte: Baujahr, quelle: CH_Bauwerksdaten.Baujahr}
      - {spalte: Gesamtlaenge_m, quelle: ASB-ING_2013.Gesamtlaenge}
      - {spalte: Gesamtbreite_m, quelle: ASB-ING_2013.Gesamtbreite}
  - blatt: Lager
    klassen: [IfcBearing]
    spalten:
      - {spalte: GlobalId, quelle: GlobalId}
      - {spalte: Name, quelle: Name}
      - {spalte: Lagertyp, quelle: BFH_Minor.Lagertyp}
      - {spalte: Lieferant, quelle: BFH_Minor.Lieferant}
      - {spalte: Modell, quelle: BFH_Minor.Modell}
      - {spalte: Einbaujahr, quelle: BFH_Minor.Einbaujahr}

"""

def lade_yaml(rel, default):
    p = REPO / rel
    if p.is_file():
        print("Aus Repo:", rel)
        return yaml.safe_load(p.read_text(encoding="utf-8"))
    print("Nicht im Repo, eingebetteter Standard:", rel)
    return yaml.safe_load(default)

def wert_auf(x):
    return JAHR if x == "aktuelles_jahr" else x

def ids_aus_regeln(cfg):
    i = ids.Ids(title=cfg.get("titel", "IDS Team 1"), date=datetime.now().strftime("%Y-%m-%d"))
    for s in cfg["spezifikationen"]:
        spez = ids.Specification(name=f'{s["id"]} {s["name"]}', ifcVersion=[cfg.get("ifcVersion", "IFC4X3_ADD2")], minOccurs=1,
                                 identifier=s["id"])
        spez.applicability.append(ids.Entity(name=s["klasse"].upper()))
        for a in s["anforderungen"]:
            opts = {}
            if a.get("erlaubte_werte"):
                opts["enumeration"] = [str(v) for v in a["erlaubte_werte"]]
            if a.get("muster"):
                opts["pattern"] = a["muster"]
            if "min" in a: opts["minInclusive"] = str(wert_auf(a["min"]))
            if "max" in a: opts["maxInclusive"] = str(wert_auf(a["max"]))
            basis = "integer" if a.get("datentyp") == "IFCINTEGER" else "string"
            wert = ids.Restriction(options=opts, base=basis) if opts else None
            spez.requirements.append(ids.Property(propertySet=a["pset"], baseName=a["name"], value=wert,
                                                  dataType=a.get("datentyp"), cardinality="required"))
        i.specifications.append(spez)
    return i

REGELN = lade_yaml(REGELN_DATEI, DEFAULT_REGELN)
MAPPING = lade_yaml(MAPPING_DATEI, DEFAULT_MAPPING)

if IDS_DATEI and (REPO / IDS_DATEI).is_file():
    IDS = ids.open(str(REPO / IDS_DATEI))
    print("IDS direkt geladen:", IDS_DATEI)
else:
    IDS = ids_aus_regeln(REGELN)
    IDS.to_xml(str(OUT / "IDS_Team1_erzeugt.ids"))
    print("IDS aus YAML-Regeln erzeugt:", len(IDS.specifications), "Spezifikationen")

## 5 · Prüfung (Schritt 6): IDS und Zusatzprüfungen

In [ ]:
import pandas as pd
import ifcopenshell.util.element as E
from ifctester import reporter

IDS.validate(modell)
res = reporter.Json(IDS).report()

zeilen = []
for sp in res["specifications"]:
    for rq in sp["requirements"]:
        for status, key in (("bestanden", "passed_entities"), ("durchgefallen", "failed_entities")):
            for e in rq[key]:
                zeilen.append({"Regel": sp["name"].split(" ")[0], "Spezifikation": sp["name"], "Anforderung": rq["label"],
                               "GlobalId": e["global_id"], "Klasse": e["class"], "Name": e["name"],
                               "Ergebnis": status, "Schweregrad": "Fehler",
                               "Grund": "" if status == "bestanden" else str(e.get("reason", rq["description"]))})

_psets = {}
def psets(el):
    if el.id() not in _psets:
        _psets[el.id()] = E.get_psets(el)
    return _psets[el.id()]

def wert(el, quelle):
    if quelle in ("GlobalId", "Name", "Tag"): return getattr(el, quelle, None)
    if quelle == "Klasse": return el.is_a()
    pset, prop = quelle.split(".", 1)
    return psets(el).get(pset, {}).get(prop)

def leer(v):
    return v is None or (isinstance(v, str) and v.strip() == "")

for z in REGELN.get("zusatzpruefungen", []):
    elemente = [e for k in z["klassen"] for e in modell.by_type(k)]
    if z["typ"] == "name_eindeutig":
        from collections import Counter
        zaehler = Counter(e.Name for e in elemente if not leer(e.Name))
        for e in elemente:
            ok = (not leer(e.Name)) and zaehler[e.Name] == 1
            zeilen.append({"Regel": z["id"], "Spezifikation": f'{z["id"]} {z["name"]}', "Anforderung": "Name",
                           "GlobalId": e.GlobalId, "Klasse": e.is_a(), "Name": e.Name, "Ergebnis": "bestanden" if ok else "durchgefallen",
                           "Schweregrad": z["schweregrad"], "Grund": "" if ok else "Name fehlt oder kommt mehrfach vor"})
    elif z["typ"] == "einbaujahr_nach_baujahr":
        baujahre = [wert(b, z["baujahr"]) for b in modell.by_type("IfcBridge")]
        baujahre = [b for b in baujahre if isinstance(b, (int, float))]
        if baujahre:
            for e in elemente:
                ej = wert(e, z["einbaujahr"])
                if not isinstance(ej, (int, float)): continue
                ok = ej >= min(baujahre)
                zeilen.append({"Regel": z["id"], "Spezifikation": f'{z["id"]} {z["name"]}', "Anforderung": z["einbaujahr"],
                               "GlobalId": e.GlobalId, "Klasse": e.is_a(), "Name": e.Name, "Ergebnis": "bestanden" if ok else "durchgefallen",
                               "Schweregrad": z["schweregrad"], "Grund": "" if ok else f"Einbaujahr {ej} liegt vor Baujahr {min(baujahre)}"})

ERG = pd.DataFrame(zeilen)
uebersicht = ERG.groupby(["Regel", "Schweregrad", "Ergebnis"]).size().unstack(fill_value=0)
print(uebersicht.to_string())

## 6 · Entscheid (Schritt 7): Daten vollständig?

In [ ]:
maengel = ERG[(ERG.Ergebnis == "durchgefallen")].copy()
fehler  = maengel[maengel.Schweregrad == "Fehler"]
warnung = maengel[maengel.Schweregrad == "Warnung"]
VOLLSTAENDIG = fehler.empty

print("=" * 60)
print("DATEN VOLLSTÄNDIG:", "JA → weiter zu Schritt 8 (Stichprobe)" if VOLLSTAENDIG
      else f"NEIN → {len(fehler)} Fehler, BCF/Mängelliste an Projektverfasser:in, IFC korrigieren und erneut einreichen")
if len(warnung): print(f"Warnungen: {len(warnung)} (blockieren die Abnahme nicht)")
print("=" * 60)
if not maengel.empty:
    display(maengel[["Regel", "Schweregrad", "GlobalId", "Klasse", "Name", "Grund"]])

## 7 · Berichte: HTML-Prüfbericht, Mängelliste (CSV, BCF)

In [ ]:
r = reporter.Html(IDS); r.report(); r.to_file(str(OUT / "pruefbericht.html"))
maengel.to_csv(OUT / "maengelliste.csv", index=False, sep=";", encoding="utf-8-sig")
ERG.to_csv(OUT / "alle_pruefergebnisse.csv", index=False, sep=";", encoding="utf-8-sig")
try:
    b = reporter.Bcf(IDS); b.report(); b.to_file(str(OUT / "maengel.bcf"))
    print("BCF erzeugt (enthält die IDS-Mängel; Zusatzprüfungen stehen nur in der CSV)")
except Exception as ex:
    print("BCF nicht erzeugt:", repr(ex))
print("Geschrieben:", ", ".join(sorted(p.name for p in OUT.iterdir())))

## 8 · Stichprobe für den Fachplaner (Schritt 8, Beurteilung manuell)

In [ ]:
import random
sp = REGELN.get("stichprobe", {})
pool = [e for k in sp.get("klassen", []) for e in modell.by_type(k)]
rng = random.Random(sp.get("seed", 1))
auswahl = rng.sample(pool, min(sp.get("anzahl", 3), len(pool))) if pool else []
STICHPROBE = pd.DataFrame([{"GlobalId": e.GlobalId, **{f: wert(e, f) for f in sp.get("felder", [])}} for e in auswahl])
STICHPROBE["Plausibel (ja/nein)"] = ""
STICHPROBE["Bemerkung"] = ""
STICHPROBE.to_csv(OUT / "stichprobe_fachplaner.csv", index=False, sep=";", encoding="utf-8-sig")
display(STICHPROBE)

## 9 · Export Stammdaten (Schritt 10)

In [ ]:
def baue_tabellen(mapping):
    blaetter = {}
    for t in mapping["tabellen"]:
        elemente = [e for k in t["klassen"] for e in modell.by_type(k)]
        blaetter[t["blatt"]] = pd.DataFrame([{s["spalte"]: wert(e, s["quelle"]) for s in t["spalten"]} for e in elemente])
    return blaetter

if VOLLSTAENDIG or EXPORT_BEI_MAENGELN:
    BLAETTER = baue_tabellen(MAPPING)
    with pd.ExcelWriter(OUT / "stammdaten.xlsx") as xw:
        for name, df in BLAETTER.items():
            df.to_excel(xw, sheet_name=name, index=False)
            df.to_csv(OUT / f"stammdaten_{name}.csv", index=False, sep=";", encoding="utf-8-sig")
    if not VOLLSTAENDIG:
        print("ACHTUNG: Export trotz Mängeln (EXPORT_BEI_MAENGELN = True), nicht zur Übernahme freigeben.")
    for name, df in BLAETTER.items():
        print(f"{name}: {len(df)} Zeilen"); display(df.head(10))
else:
    BLAETTER = baue_tabellen(MAPPING)
    print("Export übersprungen: Daten nicht vollständig (Workflow Schritt 7 = Nein).")

## 10 · Dashboard: Vollständigkeit in % je Bauteilgruppe

In [ ]:
import matplotlib.pyplot as plt
kennzahlen, fehlend = [], []
for g in REGELN.get("gruppen", []):
    elemente = [e for k in g["klassen"] for e in modell.by_type(k)]
    soll = ist = 0
    for e in elemente:
        for feld in g["erforderlich"]:
            soll += 1
            if leer(wert(e, feld)): fehlend.append({"Gruppe": g["name"], "GlobalId": e.GlobalId, "Name": e.Name, "Fehlende Angabe": feld})
            else: ist += 1
    kennzahlen.append({"Bauteilgruppe": g["name"], "Bauteile": len(elemente), "Angaben Soll": soll, "Angaben vorhanden": ist,
                       "Vollständigkeit %": round(100 * ist / soll, 1) if soll else None})
DASH = pd.DataFrame(kennzahlen)
FEHLEND = pd.DataFrame(fehlend, columns=["Gruppe", "GlobalId", "Name", "Fehlende Angabe"])

fig, ax = plt.subplots(figsize=(8, 1.2 + 0.8 * len(DASH)))
ax.barh(DASH["Bauteilgruppe"], DASH["Vollständigkeit %"].fillna(0), color="#2a6f97")
ax.set_xlim(0, 100); ax.set_xlabel("Vollständigkeit der geforderten Angaben [%]"); ax.invert_yaxis()
for y, v in enumerate(DASH["Vollständigkeit %"]): ax.text(min((v or 0) + 1, 88), y, f"{v} %", va="center")
ax.set_title(f"Vollständigkeit je Bauteilgruppe · {MODELL.name}")
plt.tight_layout(); fig.savefig(OUT / "dashboard_vollstaendigkeit.png", dpi=150); plt.show()

display(DASH)
print("Fehlende Angaben:", len(FEHLEND)); display(FEHLEND)
if "Bauwerk" in BLAETTER:
    print("Übersicht Bauwerksdaten"); display(BLAETTER["Bauwerk"].T.rename(columns=lambda c: "Wert"))
DASH.to_csv(OUT / "dashboard_vollstaendigkeit.csv", index=False, sep=";", encoding="utf-8-sig")
FEHLEND.to_csv(OUT / "fehlende_angaben.csv", index=False, sep=";", encoding="utf-8-sig")

## 11 · Ergebnis bündeln

In [ ]:
zip_pfad = shutil.make_archive("pruefergebnis", "zip", str(OUT))
print("Ergebnis-Paket:", zip_pfad)
print("Status:", "VOLLSTÄNDIG" if VOLLSTAENDIG else f"MÄNGEL ({len(fehler)} Fehler)", "| Modell:", MODELL.name)
try:
    from google.colab import files
    files.download(zip_pfad)
except ImportError:
    pass